In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
from unsloth import FastModel
import torch

In [ ]:
model_name = "unsloth/gemma-3-12b-it-unsloth-bnb-4bit"

model, tokenizer = FastModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [4]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [5]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_data_formats

dataset = load_dataset("csv", data_files="progressive_train_80.csv")["train"]
dataset = standardize_data_formats(dataset)


In [7]:
def apply_chat_template(example):
    return {
        "text": tokenizer.apply_chat_template(
            [{"role": "user", "content": str(example["input"])},
             {"role": "assistant", "content": str(example["output"])}],
            tokenize=False
        )
    }

dataset = dataset.map(apply_chat_template)


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,  # No accumulation, update after every example
        warmup_steps = 0,
        num_train_epochs = 1,
        max_steps = 2000,  # Or remove it to let epochs control training
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        save_steps=500,
        save_strategy = "steps"
    )
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

In [14]:
torch.cuda.empty_cache()  # Clear GPU memory

In [ ]:
trainer_stats = trainer.train()

In [14]:
new_model_local = "Gemma-3-12B-it-FirstResponder_progressive_train_80_500steps"
model.save_pretrained(new_model_local) # Local saving
tokenizer.save_pretrained(new_model_local)

['Gemma-3-12B-it-FirstResponder_progressive_train_80_500steps/processor_config.json']